In [1]:
import pandas as pd
import json
import os
from dataclasses import dataclass
import re

import helper as analytics
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import tqdm

ModuleNotFoundError: No module named 'helper'

## NPlaug examples

In [3]:
nplaug_data = pd.read_csv("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_all_augmentation_nplaug_matching_examples_v2.csv")
nplaug_data.head()

,prompt,completion,id
0,Do the two product descriptions refer to the s...,No,41987114#35410623_original
1,Do the two product descriptions refer to the s...,No,39417365#79821099_original
2,Do the two product descriptions refer to the s...,No,5938532#27021728_original
3,Do the two product descriptions refer to the s...,No,77338417#85033463_original
4,Do the two product descriptions refer to the s...,No,42341716#49092761_original


In [6]:
nplaug_data[nplaug_data["completion"] == "Yes"]

,prompt,completion,id
5,Do the two product descriptions refer to the s...,Yes,18960736#24268406_original
6,Do the two product descriptions refer to the s...,Yes,20060104#29467728_left_augmented_right_original
11,Do the two product descriptions refer to the s...,Yes,25593960#41191422_left_augmented_right_original
12,Do the two product descriptions refer to the s...,Yes,24742825#97547150_left_original_right_augmented
13,Do the two product descriptions refer to the s...,Yes,2491460#75249284_left_augmented_right_original
...,...,...,...
3912,Do the two product descriptions refer to the s...,Yes,42604445#33365078_left_original_right_augmented
3913,Do the two product descriptions refer to the s...,Yes,66047716#47613152_original
3915,Do the two product descriptions refer to the s...,Yes,93836358#71957793_left_augmented_right_original
3917,Do the two product descriptions refer to the s...,Yes,24224044#47282882_left_augmented_right_original


In [8]:
# find all records that contain a certain id string and show full prompt text
pd.set_option('display.max_colwidth', None)  # Remove column width limits
nplaug_data[nplaug_data["id"].str.contains("18960736#24268406")]


,prompt,completion,id
5,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi Dual-Band WiFi PoE Access Point UAP-AC-HD - 5 Pack'. Entity 2: 'UniFi AP ac HD ROW 5Pk'.,Yes,18960736#24268406_original
1710,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti U niFi Dual - Band WiFi PoE Acc ess Point UAP - AC - HD - 5 Pack'. Entity 2: 'UniFi AP ac HD ROW 5Pk'.,Yes,18960736#24268406_left_augmented_right_original
2377,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti U niFi Dual - Band WiFi PoE Acc ess Point UAP - AC - HD - 5 Pack'. Entity 2: 'UniFi AP _ HD ROW 5Pk'.,Yes,18960736#24268406_both_augmented
3799,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi Dual-Band WiFi PoE Access Point UAP-AC-HD - 5 Pack'. Entity 2: 'UniFi AP _ HD ROW 5Pk'.,Yes,18960736#24268406_left_original_right_augmented


## Swapping

In [9]:
swapping_25_df = pd.read_csv("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_explanations_40_swapped_matching_examples_0_25_permutations_v2.csv")
swapping_25_df.head()

,prompt,completion,id
0,"Do the two product descriptions refer to the same real-world product? Entity 1: 'HDD 35 4TB Seagate IronWolf Pro NAS ST4000NE001'. Entity 2: 'HD 3,5 4TB 7200RPM IRONWOLF PRO 128 MB SATA3 SEAGATE'.",Yes,14654897#36425270
1,Do the two product descriptions refer to the same real-world product? Entity 1: 'Buy Quality Replica Omega Seamaster Planet Ocean 600M Steel Chronometer Chronograph Watch 215.30.46.51.01.001'. Entity 2: 'GIGABYTE Radeon RX 5500 XT OC - 4GB GDDR6 RAM - Grafikkort'.,No,31531912#60397145
2,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UVC-G3-FLEX-3 UniFi Protect G3 FLEX Camera 3-Pack'. Entity 2: 'AAA Replica Omega Seamaster Planet Ocean 600M Chronograph Stainless Steel Blue Watch 215.30.46.51.03.001'.,No,44557157#90806148
3,Do the two product descriptions refer to the same real-world product? Entity 1: 'Brother HL-L6300DW Business Laser Printer for Mid-Size Workgroups -HL-L6300DW'. Entity 2: 'Epson T6923 Ultrachrome XD rautt Ink 110ml'.,No,49605449#36985401
4,Do the two product descriptions refer to the same real-world product? Entity 1: 'KINGSTON 64GB USB 3.0 DataTraveler SE9 G2 (Kovový) DTSE9G2/64GB'. Entity 2: 'Buy Quality Replica Tag Heuer Monaco Steve McQueen Chronograph Watch CAW211P.FC6356'.,No,3024917#70174967


In [11]:
swapping_25_df[swapping_25_df["completion"] == "Yes"]

,prompt,completion,id
0,"Do the two product descriptions refer to the same real-world product? Entity 1: 'HDD 35 4TB Seagate IronWolf Pro NAS ST4000NE001'. Entity 2: 'HD 3,5 4TB 7200RPM IRONWOLF PRO 128 MB SATA3 SEAGATE'.",Yes,14654897#36425270
27,Do the two product descriptions refer to the same real-world product? Entity 1: 'Mavic 2 Part11 Car Charger'. Entity 2: 'DJI Mavic 2 Car Charger'.,Yes,15306268#24574240
28,"Do the two product descriptions refer to the same real-world product? Entity 1: 'Johnnie Walker - Blue Label 70cl'. Entity 2: 'Johnnie Walker Blue Label 0,7 ltr.'.",Yes,24351127#81212084
29,Do the two product descriptions refer to the same real-world product? Entity 1: 'Jabra EVOLVE 80 MS Stereo (7899-823-109)'. Entity 2: 'Jabra Evolve 80 UC stereo SKype for Business'.,Yes,13142970#95282914
30,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi Video Camera G4 Pro (UVC-G4-PRO)'. Entity 2: 'Ubiquiti UniFi Protect G4-PRO Camera'.,Yes,21670568#79415655
...,...,...,...
5229,"Do the two product descriptions refer to the same real-world product? Entity 1: 'EPSON (T1285) Multipack Fox (N, C, M, J) '. Entity 2: 'Epson róka Multipack \""Renard\"" (T1285) - Encre DURABrite Ultra T1281,T1282,T1283,T1284 C13T12854010'.",Yes,28282387#18309559_swapped_colors_product code_series
5230,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi UVC-G3-BULLET (Formerly UVC-G3-AF) Video Camera'. Entity 2: 'Ubiquiti G3 Bullet UniFi Video Camera G3 1080p PoE IP Camera'.,Yes,80529811#16871572_swapped_model
5231,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi G3 Bullet Video Camera 1080p IP Camera'. Entity 2: 'Ubiquiti UVC-G3-BULLET (Formerly UVC-G3-AF) UniFi Video Camera G3 PoE '.,Yes,80529811#16871572_swapped_connectivity_resolution
5232,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi UVC-G3-BULLET (Formerly UVC-G3-AF) Video Camera 1080p IP Camera'. Entity 2: 'Ubiquiti G3 Bullet UniFi Video Camera G3 PoE '.,Yes,80529811#16871572_swapped_connectivity_model_resolution


In [12]:
swapping_25_df[swapping_25_df["id"].str.contains("80529811#16871572")]

,prompt,completion,id
2497,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi G3 Bullet Video Camera'. Entity 2: 'Ubiquiti UVC-G3-BULLET (Formerly UVC-G3-AF) UniFi Video Camera G3 1080p PoE IP Camera'.,Yes,80529811#16871572
5230,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi UVC-G3-BULLET (Formerly UVC-G3-AF) Video Camera'. Entity 2: 'Ubiquiti G3 Bullet UniFi Video Camera G3 1080p PoE IP Camera'.,Yes,80529811#16871572_swapped_model
5231,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi G3 Bullet Video Camera 1080p IP Camera'. Entity 2: 'Ubiquiti UVC-G3-BULLET (Formerly UVC-G3-AF) UniFi Video Camera G3 PoE '.,Yes,80529811#16871572_swapped_connectivity_resolution
5232,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi UVC-G3-BULLET (Formerly UVC-G3-AF) Video Camera 1080p IP Camera'. Entity 2: 'Ubiquiti G3 Bullet UniFi Video Camera G3 PoE '.,Yes,80529811#16871572_swapped_connectivity_model_resolution
5233,Do the two product descriptions refer to the same real-world product? Entity 1: 'Ubiquiti UniFi G3 Bullet Video Camera 1080p IP Camera PoE'. Entity 2: 'Ubiquiti UVC-G3-BULLET (Formerly UVC-G3-AF) UniFi Video Camera G3 '.,Yes,80529811#16871572_swapped_connectivity_power_resolution
